In [4]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
model_paths = {
    'alexnet': 'models/alexnet.pth',
    'vgg16': 'models/vgg16.pth',
    'googlenet': 'models/googlenet.pth',
    'resnet50': 'models/resnet50.pth',
    'efficientnet_b0': 'models/efficientnet_b0.pth'
}

In [5]:
data_dir = "/Users/allikhankoshamet/Desktop/dl_project/dl_new_dataset"

In [ ]:
data_transforms = {
        'val': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
}

In [8]:
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), data_transforms['val'])
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [9]:
class_names = test_dataset.classes
num_classes = len(class_names)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [11]:
print(f"✅ Test set : {len(test_dataset)} img, {num_classes} class's")
print(f"Class's: {class_names}\n")

✅ Test set : 60 img, 15 class's
Class's: ['battery', 'copybook', 'espnder', 'headphones', 'highlighter', 'marker', 'mouse', 'napcin', 'notebook', 'paper holder', 'pen', 'pencile', 'stepler', 'sticker', 'usb']



In [12]:
results = []

In [13]:
for name, path in model_paths.items():
    print(f"🔄 Matching {name.upper()}...")

    if name == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == 'googlenet':
        model = models.googlenet(weights=None, aux_logits=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'resnet50':
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=name):
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    
    results.append({
        'Model': name.upper(),
        'Test Accuracy': round(acc, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-score': round(f1, 4),
        'Model Size (MB)': round(size_mb, 2)
    })

🔄 Matching ALEXNET...


alexnet: 100%|██████████| 8/8 [00:07<00:00,  1.01it/s]


🔄 Matching VGG16...


vgg16: 100%|██████████| 8/8 [00:07<00:00,  1.04it/s]
/opt/homebrew/lib/python3.10/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


🔄 Matching GOOGLENET...


googlenet: 100%|██████████| 8/8 [00:08<00:00,  1.03s/it]


🔄 Matching RESNET50...


resnet50: 100%|██████████| 8/8 [00:07<00:00,  1.06it/s]


🔄 Matching EFFICIENTNET_B0...


efficientnet_b0: 100%|██████████| 8/8 [00:07<00:00,  1.03it/s]


In [14]:
df = pd.DataFrame(results)
print("\n" + "="*80)
print("🏆")
print("="*80)
print(df.round(4))

# Сохраняем в файлы
df.to_csv('model_comparison.csv', index=False)
df.to_excel('model_comparison.xlsx', index=False)
print("All dobe")


🏆
             Model  Test Accuracy  Precision  Recall  F1-score  \
0          ALEXNET         0.9000     0.9117  0.9000    0.9001   
1            VGG16         0.9500     0.9688  0.9500    0.9509   
2        GOOGLENET         0.8667     0.8649  0.8667    0.8497   
3         RESNET50         1.0000     1.0000  1.0000    1.0000   
4  EFFICIENTNET_B0         0.8667     0.8558  0.8667    0.8458   

   Model Size (MB)  
0           217.69  
1           512.41  
2            21.59  
3            90.09  
4            15.64  
All dobe
